# 18 — Whole-week features and staged XGBoost/LightGBM blend

**Henrik Remmen (541747), Group 119 — [119] SINTEF Databreach**

Current reported best: notebook 17 blend **0.99005** (XGB 0.98996, LGBM 0.98968).
This is a new experiment, not a guarantee of beating 0.99562. Keep your current best CSV.

## Hypotheses and controls

1. The successful models reduce the price/inflow trajectory to many local summaries.
   Add each day's price mean/peak relative to the current hour, six fixed DCT shape
   coefficients, contiguous competing-price runs, each day's profitable/unprofitable
   margin and source inflow, and the concentration/timing of profitable opportunities.
   These are 54 input-only features per plant. No future observed storage or labels.
2. The old per-unit tree counts were frozen from earlier experiments. They may underfit
   the richer features/larger training history. Fit up to 2x those counts, retaining
   predictions at 1x, 1.5x and 2x. Keep depth/leaves, learning rate, regularization and
   seeds unchanged. This distinguishes extra boosting from the new feature group.

2021 selects the feature set and multiplier for each backend, then one global XGB
weight from {0.25, 0.5, 0.75}. 2022 checks that frozen pair against the exact v4 50/50
control. No per-generator weight search, no label smoothing, no hard ON/OFF output.
Both years have informed earlier development, so neither is a pristine holdout.
Daily case horizons are purged at year boundaries. Public scores do not enter selection.

## Run

Restart the kernel and run all cells. A = development, B = historical audit,
C = final training and CSV creation. Final cell exports a self-contained professor
notebook. That short notebook embeds all code and trains from raw data, without
requiring any models, artifacts, `src/`, or other notebooks. It always uses <=4 CPU
threads, with cache reads disabled. Use the project's pinned requirements.

Local mode defaults to 12 threads, sequential float32 fits, BLAS one thread.
Completed per-generator tasks resume after an interruption; the interrupted fit
restarts. 2022/final v4 controls can reuse notebook-17 predictions only if raw data,
source signature, parameters, rounds, case IDs and package versions match.
Native notebook-18 checkpoints also include thread count and exact source signature.
Changing thread count invalidates those native caches to avoid hidden assumptions.

Research has <=140 unit fits, not 420: each development fit yields three prefixes.
Final-only reproduction has exactly 28 fits, at most twice the old tree counts.
No GPU or parallel model copies. Training runtime/peak RAM have NOT been benchmarked;
cold-test the selected short notebook on four cores against 12 hours / 32 GB.
If the audit fails, the new CSV remains explicitly experimental; it does not silently
replace itself with the older blend. A selected 1x v4 pair can legitimately reproduce
the old model if the new alternatives lose.

Only small diagnostic probes and tiny API tests were run while building this notebook.
No full-scale candidate training was performed by the assistant.

In [1]:
# FIRST CODE CELL: execution resources only; identical statistical settings in both profiles.
import os
RUN_PROFILE = 'local'  # 'professor': <=4 CPU threads, no restored predictions
LOCAL_CPU_THREADS = 8  # Try 8 if the Mac is unresponsive; 12 is not benchmarked as optimal.
if RUN_PROFILE not in ('local','professor'):
    raise ValueError('Unknown profile')
THREADS = min(LOCAL_CPU_THREADS if RUN_PROFILE=='local' else 4, os.cpu_count() or 1)
if THREADS < 1:
    raise ValueError('Need at least one thread')
USE_CHECKPOINTS = RUN_PROFILE=='local'
for key in ('OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','VECLIB_MAXIMUM_THREADS','NUMEXPR_NUM_THREADS'):
    os.environ[key]='1'
os.environ['OMP_NUM_THREADS']=str(THREADS)
print(f'{RUN_PROFILE}: {THREADS} CPU threads; one model at a time; restore={USE_CHECKPOINTS}')

local: 8 CPU threads; one model at a time; restore=True


In [2]:
from pathlib import Path
import os
import time
from IPython.display import display

NOTEBOOK_STARTED = time.perf_counter()
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/kernel').is_dir() and (p / 'prediction_mapping.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Place notebook beside the supplied data/ directory and mapping files.')
SOURCE_DIGEST = '287e31324a2248f4584cae3540ec1f4d46579780493d1d4f06ef7b02ca1a93fb'
V4_SOURCE_DIGEST = '94b35b1ac9206f02683466ec302e8ee1e1ac66ea92847b5ac549639a63071e7b'

# Existing models are frozen controls. Their counts were determined in prior development.
BASELINE_ROUNDS = {
 'xgb_v2': dict(zip(
  ['Hogga_G1','Tokke_G1','Tokke_G2','Tokke_G3','Tokke_G4','Lio_G1','Byrte_G1',
   'Vinje_G1','Vinje_G2','Vinje_G3','Haukeli_G1','Songa_G1','Kjela_G1','Vesle Kjela_G1'],
  [410,450,854,587,693,249,292,658,404,607,223,510,326,372])),
 'lgb_v2': dict(zip(
  ['Hogga_G1','Tokke_G1','Tokke_G2','Tokke_G3','Tokke_G4','Lio_G1','Byrte_G1',
   'Vinje_G1','Vinje_G2','Vinje_G3','Haukeli_G1','Songa_G1','Kjela_G1','Vesle Kjela_G1'],
  [398,534,387,458,410,386,441,852,299,873,330,465,393,177]))}
xgb_parameters = dict(objective='binary:logistic', eval_metric='auc', learning_rate=.03,
    max_depth=7, min_child_weight=50, subsample=.85, colsample_bytree=.85,
    reg_alpha=.1, reg_lambda=5., tree_method='hist', max_bin=256, device='cpu', random_state=42)
lgb_parameters = dict(objective='binary', metric='None', learning_rate=.03, num_leaves=31,
    max_depth=-1, min_data_in_leaf=150, feature_fraction=.85, bagging_fraction=.85,
    bagging_freq=1, lambda_l1=.1, lambda_l2=5., max_bin=255, device_type='cpu',
    deterministic=True, force_col_wise=True, seed=42, verbosity=-1)
ROUNDS = {'xgb':BASELINE_ROUNDS['xgb_v2'], 'lgb':BASELINE_ROUNDS['lgb_v2']}
SPECS = {
    'xgb':dict(backend='xgboost',features='competition',max_rounds=2000,parameters=xgb_parameters),
    'lgb':dict(backend='lightgbm',features='competition',max_rounds=1800,parameters=lgb_parameters),
}

## pipeline — definitions only

In [3]:
"""Shared, target-safe data and feature pipeline used by experiment notebooks.

The module contains no model training.  It exists so CatBoost and XGBoost
experiments use exactly the same inputs, folds, metric, and submission layout.
"""

from dataclasses import dataclass
from pathlib import Path
import re

import numpy as np
import pandas as pd
import yaml
from sklearn.metrics import roc_auc_score


HORIZON = 168
RANDOM_SEED = 42


@dataclass
class ProjectData:
    root: Path
    X_all: pd.DataFrame
    target_cube: np.ndarray
    units: list[str]
    train_cases: pd.DataFrame
    test_cases: pd.DataFrame
    sample_submission: pd.DataFrame
    feature_sets: dict[str, list[str]]
    feature_group: dict[str, str]
    generator_features: pd.DataFrame
    folds: list[dict]

    @property
    def n_train_cases(self):
        return len(self.train_cases)

    @property
    def n_test_cases(self):
        return len(self.test_cases)

    @property
    def n_all_cases(self):
        return self.n_train_cases + self.n_test_cases


def read_time_csv(path, time_column="date"):
    frame = pd.read_csv(path)
    frame[time_column] = pd.to_datetime(frame[time_column], utc=True)
    return frame


def rows_for_cases(case_ids):
    case_ids = np.asarray(case_ids)
    return (
        case_ids[:, None] * HORIZON
        + np.arange(HORIZON)[None, :]
    ).ravel()


def micro_auc(y_true_cube, y_pred_cube):
    if y_true_cube.shape != y_pred_cube.shape:
        raise ValueError("Truth and prediction cubes must have identical shapes")
    return roc_auc_score(
        y_true_cube.reshape(len(y_true_cube), -1),
        y_pred_cube.reshape(len(y_pred_cube), -1),
        average="micro",
    )


def per_unit_auc(y_true_cube, y_pred_cube, units):
    return {
        unit: roc_auc_score(
            y_true_cube[:, unit_index, :].ravel(),
            y_pred_cube[:, unit_index, :].ravel(),
        )
        for unit_index, unit in enumerate(units)
    }


def make_purged_folds(train_cases, validation_years=(2020, 2021, 2022)):
    starts = pd.DatetimeIndex(train_cases["starttime"])
    folds = []
    for year in validation_years:
        validation_start = pd.Timestamp(f"{year}-01-01", tz="UTC")
        validation_end = pd.Timestamp(f"{year + 1}-01-01", tz="UTC")
        validation_cases = np.flatnonzero(
            (starts >= validation_start) & (starts < validation_end)
        )
        training_cases = np.flatnonzero(
            starts + pd.Timedelta(hours=HORIZON - 1) < validation_start
        )
        if not len(training_cases) or not len(validation_cases):
            raise ValueError(f"Empty training or validation partition for {year}")
        if not (
            starts[training_cases].max() + pd.Timedelta(hours=HORIZON - 1)
            < starts[validation_cases].min()
        ):
            raise AssertionError(f"Fold {year} is not temporally purged")
        folds.append({
            "year": year,
            "train_cases": training_cases,
            "validation_cases": validation_cases,
        })
    return folds


def make_submission(data, prediction_cube, filename):
    expected = (data.n_test_cases, len(data.units), HORIZON)
    if prediction_cube.shape != expected:
        raise ValueError(f"Expected predictions {expected}, got {prediction_cube.shape}")

    submission_data = {"Run No": data.test_cases["Run No"].to_numpy()}
    for unit_index, unit in enumerate(data.units):
        for horizon in range(HORIZON):
            submission_data[f"result_committed_{unit}_t{horizon}"] = (
                prediction_cube[:, unit_index, horizon]
            )

    submission = pd.DataFrame(submission_data)[data.sample_submission.columns]
    probabilities = submission.iloc[:, 1:].to_numpy()
    if not submission["Run No"].equals(data.sample_submission["Run No"]):
        raise AssertionError("Run No does not match sample_submission.csv")
    if not np.isfinite(probabilities).all():
        raise ValueError("Submission contains missing or non-finite predictions")
    if probabilities.min() < 0 or probabilities.max() > 1:
        raise ValueError("Submission probabilities are outside [0, 1]")

    path = data.root / "submissions" / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(path, index=False)
    return path, submission


def build_project_data(root):
    root = Path(root).resolve()
    data_dir = root / "data"

    price = read_time_csv(data_dir / "kernel" / "Historical_day_ahead_price_2015_2025.csv")
    inflow = read_time_csv(data_dir / "kernel" / "Historical_inflow_1958_2025.csv")
    volume = read_time_csv(data_dir / "kernel" / "Historical_volume_2015_2024.csv")
    water_value = read_time_csv(data_dir / "kernel" / "Synthetic_water_value_2015_2024.csv")
    min_flow = read_time_csv(data_dir / "extended" / "Constraint_min_flow.csv")
    min_volume = read_time_csv(data_dir / "extended" / "Constraint_min_volume.csv")
    targets = read_time_csv(data_dir / "kernel" / "Unit_commitment_decisions.csv", "starttime")
    prediction_mapping = read_time_csv(root / "prediction_mapping.csv", "starttime")
    sample_submission = pd.read_csv(root / "sample_submission.csv")

    with open(data_dir / "extended" / "Tokke_Vinje_topology.yaml") as stream:
        topology = yaml.safe_load(stream)

    pattern = re.compile(r"^result_committed_(.+)_t(\d+)$")
    target_columns = [column for column in targets if pattern.match(column)]
    units = list(dict.fromkeys(pattern.match(column).group(1) for column in target_columns))
    if len(units) != 14 or len(target_columns) != 14 * HORIZON:
        raise AssertionError("Expected 14 units and 168 targets per unit")
    if target_columns != list(sample_submission.columns[1:]):
        raise AssertionError("Target order differs from sample submission")

    target_cube = np.stack([
        targets[[f"result_committed_{unit}_t{hour}" for hour in range(HORIZON)]]
        .to_numpy(dtype=np.int8)
        for unit in units
    ], axis=1)
    if not set(np.unique(target_cube)).issubset({0, 1}):
        raise AssertionError("Targets must be binary")

    train_cases = targets[["Run No", "starttime"]].copy()
    test_cases = prediction_mapping[["Run No", "starttime"]].copy()
    all_cases = pd.concat([train_cases, test_cases], ignore_index=True)
    n_all_cases = len(all_cases)
    starts = pd.DatetimeIndex(all_cases["starttime"])
    horizon_hour = np.tile(np.arange(HORIZON), n_all_cases)
    target_time = pd.DatetimeIndex(
        starts.repeat(HORIZON) + pd.to_timedelta(horizon_hour, unit="h")
    )

    def numeric_indexed(frame):
        return frame.set_index("date").select_dtypes(include=np.number)

    def hourly_matrix(series):
        values = series.reindex(target_time).to_numpy()
        matrix = values.reshape(n_all_cases, HORIZON)
        if not np.isfinite(matrix).all():
            raise ValueError(
                f"Missing/non-finite hourly inputs: {(~np.isfinite(matrix)).sum()}"
            )
        return matrix

    def repeat_case(values):
        values = np.asarray(values)
        if len(values) != n_all_cases:
            raise ValueError("Case-level array has incorrect length")
        return np.repeat(values, HORIZON)

    def cyclic(values, period):
        angle = 2 * np.pi * np.asarray(values) / period
        return np.sin(angle), np.cos(angle)

    def clipped_shift(matrix, offset):
        indices = np.clip(np.arange(HORIZON) + offset, 0, HORIZON - 1)
        return matrix[:, indices]

    def centred_rolling_mean(matrix, window):
        return (
            pd.DataFrame(matrix).T
            .rolling(window=window, min_periods=1, center=True)
            .mean().T.to_numpy()
        )

    feature_data = {}
    feature_group = {}

    def add_feature(name, values, group):
        values = np.asarray(values)
        expected = n_all_cases * HORIZON
        if values.ndim != 1 or len(values) != expected:
            raise ValueError(f"{name}: expected {expected}, got {values.shape}")
        feature_data[name] = values.astype(np.float32)
        feature_group[name] = group

    hour = target_time.hour.to_numpy()
    weekday = target_time.dayofweek.to_numpy()
    month = target_time.month.to_numpy()
    for name, values, period in (
        ("hour", hour, 24),
        ("weekday", weekday, 7),
        ("month", month - 1, 12),
    ):
        sine, cosine = cyclic(values, period)
        add_feature(f"{name}_sin", sine, "calendar")
        add_feature(f"{name}_cos", cosine, "calendar")
    add_feature("horizon_hour", horizon_hour, "calendar")

    price_matrix = hourly_matrix(
        price.set_index("date")["Day-ahead price (EUR/MWh)"]
    )
    price_mean = price_matrix.mean(axis=1)
    price_std = price_matrix.std(axis=1)
    price_rank = pd.DataFrame(price_matrix).rank(axis=1, pct=True).to_numpy()
    add_feature("price", price_matrix.ravel(), "price")
    add_feature("price_week_mean", repeat_case(price_mean), "price")
    add_feature("price_week_std", repeat_case(price_std), "price")
    add_feature("price_week_percentile", price_rank.ravel(), "price")
    add_feature("price_minus_week_mean", (price_matrix - price_mean[:, None]).ravel(), "price")
    add_feature(
        "price_week_zscore",
        ((price_matrix - price_mean[:, None]) / (price_std[:, None] + 1e-6)).ravel(),
        "price_windows",
    )
    add_feature("price_top_10pct", (price_rank >= 0.90).ravel(), "price_windows")
    add_feature("price_bottom_10pct", (price_rank <= 0.10).ravel(), "price_windows")
    for offset in (1, 3, 6, 24):
        lag = clipped_shift(price_matrix, -offset)
        lead = clipped_shift(price_matrix, offset)
        add_feature(f"price_lag_{offset}h", lag.ravel(), "price_windows")
        add_feature(f"price_lead_{offset}h", lead.ravel(), "price_windows")
        add_feature(f"price_change_from_{offset}h", (price_matrix - lag).ravel(), "price_windows")
        add_feature(f"price_change_to_next_{offset}h", (lead - price_matrix).ravel(), "price_windows")
    for window in (3, 6, 12, 24, 48):
        rolling = centred_rolling_mean(price_matrix, window)
        add_feature(f"price_mean_{window}h", rolling.ravel(), "price_windows")
        add_feature(f"price_minus_mean_{window}h", (price_matrix - rolling).ravel(), "price_windows")

    reservoirs = [column for column in volume.columns if column != "date"]
    capacity = {
        name: values["max_vol"]
        for name, values in topology["model"]["reservoir"].items()
    }
    start_volume = numeric_indexed(volume).reindex(starts)[reservoirs]
    start_water = numeric_indexed(water_value).reindex(starts)[reservoirs]
    end_water = numeric_indexed(water_value).reindex(
        starts + pd.Timedelta(days=7)
    )[reservoirs]
    if start_volume.isna().any().any() or start_water.isna().any().any() or end_water.isna().any().any():
        raise ValueError("Missing case-level reservoir data")

    for reservoir in reservoirs:
        fill = start_volume[reservoir].to_numpy() / capacity[reservoir]
        water_start = start_water[reservoir].to_numpy()
        water_end = end_water[reservoir].to_numpy()
        add_feature(f"fill__{reservoir}", repeat_case(fill), "hydrology")
        add_feature(f"water_value_end__{reservoir}", repeat_case(water_end), "water_margins")
        add_feature(f"water_value_change__{reservoir}", repeat_case(water_end - water_start), "water_margins")
        add_feature(f"price_margin__{reservoir}", (price_matrix - water_end[:, None]).ravel(), "water_margins")

    system_fill = (
        start_volume.sum(axis=1).to_numpy()
        / sum(capacity[reservoir] for reservoir in reservoirs)
    )
    water_end_mean = end_water.mean(axis=1).to_numpy()
    add_feature("system_filling_ratio", repeat_case(system_fill), "hydrology")
    add_feature("water_value_end_mean", repeat_case(water_end_mean), "water_margins")
    margin = price_matrix - water_end_mean[:, None]
    add_feature("price_minus_water_value_mean", margin.ravel(), "water_margins")
    add_feature(
        "price_to_abs_water_value_ratio",
        (price_matrix / (np.abs(water_end_mean[:, None]) + 1.0)).ravel(),
        "water_margins",
    )

    inflow_values = numeric_indexed(inflow)
    inflow_matrices = {}
    for column in inflow_values:
        matrix = hourly_matrix(inflow_values[column])
        inflow_matrices[column] = matrix
        add_feature(f"inflow__{column}", matrix.ravel(), "hydrology")
        add_feature(f"inflow_week_mean__{column}", repeat_case(matrix.mean(axis=1)), "hydrology")
    total_inflow = np.sum(list(inflow_matrices.values()), axis=0)
    add_feature("inflow_total", total_inflow.ravel(), "hydrology")
    add_feature("inflow_total_week_mean", repeat_case(total_inflow.mean(axis=1)), "hydrology")
    add_feature("inflow_total_week_std", repeat_case(total_inflow.std(axis=1)), "hydrology")
    add_feature("inflow_total_24h_mean", centred_rolling_mean(total_inflow, 24).ravel(), "hydrology")
    add_feature(
        "inflow_total_cumulative_mean",
        (np.cumsum(total_inflow, axis=1) / (np.arange(HORIZON)[None, :] + 1)).ravel(),
        "hydrology",
    )

    flow_values = numeric_indexed(min_flow)
    min_volume_values = numeric_indexed(min_volume)
    flow_matrices = {}
    min_volume_matrices = {}
    for column in flow_values:
        matrix = hourly_matrix(flow_values[column])
        flow_matrices[column] = matrix
        add_feature(f"min_flow__{column}", matrix.ravel(), "constraints")
    for column in min_volume_values:
        matrix = hourly_matrix(min_volume_values[column])
        min_volume_matrices[column] = matrix
        add_feature(f"min_volume__{column}", matrix.ravel(), "constraints")
        if column in reservoirs:
            initial = start_volume[column].to_numpy()[:, None]
            add_feature(
                f"min_volume_slack__{column}",
                ((initial - matrix) / capacity[column]).ravel(),
                "constraints",
            )

    flow_active_count = np.sum([matrix > 0 for matrix in flow_matrices.values()], axis=0)
    volume_active_count = np.sum([matrix > 0 for matrix in min_volume_matrices.values()], axis=0)
    total_min_flow = np.sum(list(flow_matrices.values()), axis=0)
    total_min_volume = np.sum(list(min_volume_matrices.values()), axis=0)
    add_feature("min_flow_active_count", flow_active_count.ravel(), "constraints")
    add_feature("min_volume_active_count", volume_active_count.ravel(), "constraints")
    add_feature("min_flow_total", total_min_flow.ravel(), "constraints")
    add_feature("min_volume_total", total_min_volume.ravel(), "constraints")

    variable_flow_columns = [
        column for column in flow_values
        if 0 < (flow_values[column] > 0).mean() < 1
    ]
    variable_volume_columns = [
        column for column in min_volume_values
        if 0 < (min_volume_values[column] > 0).mean() < 1
    ]
    variable_active = np.zeros((n_all_cases, HORIZON), dtype=bool)
    for column in variable_flow_columns:
        variable_active |= flow_matrices[column] > 0
    for column in variable_volume_columns:
        variable_active |= min_volume_matrices[column] > 0
    add_feature("variable_constraint_active", variable_active.ravel(), "constraints")

    add_feature("margin_x_system_fill", (margin * system_fill[:, None]).ravel(), "interactions")
    add_feature("price_rank_x_system_fill", (price_rank * system_fill[:, None]).ravel(), "interactions")
    add_feature("margin_x_total_inflow", (margin * total_inflow).ravel(), "interactions")
    add_feature("price_rank_x_hour_sin", price_rank.ravel() * feature_data["hour_sin"], "interactions")
    add_feature("price_rank_x_hour_cos", price_rank.ravel() * feature_data["hour_cos"], "interactions")
    add_feature("margin_x_constraint", margin.ravel() * variable_active.ravel(), "interactions")

    X_all = pd.DataFrame(feature_data)
    constants = [
        column for column in X_all
        if X_all[column].nunique(dropna=False) <= 1
    ]
    X_all = X_all.drop(columns=constants)
    if X_all.isna().any().any() or not np.isfinite(X_all.to_numpy()).all():
        raise ValueError("Final feature matrix contains invalid values")

    baseline = [
        "hour_sin", "hour_cos", "weekday_sin", "weekday_cos",
        "month_sin", "month_cos", "horizon_hour", "price",
        "price_week_mean", "price_week_std", "price_week_percentile",
        "price_minus_week_mean", "water_value_end_mean",
        "price_minus_water_value_mean", "system_filling_ratio",
        "inflow_total", "inflow_total_24h_mean", "min_flow_total",
        "min_volume_total", "variable_constraint_active",
    ]

    def grouped(name):
        return [
            feature for feature, group in feature_group.items()
            if group == name and feature in X_all.columns
        ]

    def unique_existing(names):
        return list(dict.fromkeys(name for name in names if name in X_all.columns))

    feature_sets = {
        "baseline": unique_existing(baseline),
        "baseline_plus_price": unique_existing(baseline + grouped("price_windows")),
        "baseline_plus_price_water": unique_existing(
            baseline + grouped("price_windows") + grouped("water_margins")
        ),
        "baseline_plus_price_water_hydrology": unique_existing(
            baseline + grouped("price_windows") + grouped("water_margins")
            + grouped("hydrology")
        ),
        "full_engineered": unique_existing(
            baseline + grouped("price_windows") + grouped("water_margins")
            + grouped("hydrology") + grouped("constraints")
            + grouped("interactions")
        ),
    }

    generator_rows = []
    generator_config = topology["model"]["generator"]
    for unit in units:
        config = generator_config[unit]
        plant = unit.rsplit("_G", 1)[0]
        number_match = re.search(r"_G(\d+)$", unit)
        generator_rows.append({
            "unit": unit,
            "plant": plant,
            "generator_number": int(number_match.group(1)) if number_match else 1,
            "p_min": config.get("p_min", np.nan),
            "p_max": config.get("p_max", np.nan),
            "p_nom": config.get("p_nom", np.nan),
            "minimum_output_fraction": config.get("p_min", np.nan) / config.get("p_max", np.nan),
            "start_cost": config.get("startcost_const", 0.0),
            "stop_cost": config.get("stopcost_const", 0.0),
            "penstock": str(config.get("penstock", "unknown")),
        })
    generator_features = pd.DataFrame(generator_rows)
    generator_features["units_in_plant"] = (
        generator_features.groupby("plant")["unit"].transform("count")
    )

    return ProjectData(
        root=root,
        X_all=X_all,
        target_cube=target_cube,
        units=units,
        train_cases=train_cases,
        test_cases=test_cases,
        sample_submission=sample_submission,
        feature_sets=feature_sets,
        feature_group=feature_group,
        generator_features=generator_features,
        folds=make_purged_folds(train_cases),
    )

## physics — definitions only

In [4]:
"""Deterministic input-only features for the next hydropower experiments.

No fitted transforms, labels, predicted/observed future storage, or solver outputs.
Water budgets deliberately exclude unknown dispatch and are explicitly proxies.
"""

import numpy as np
import pandas as pd
import yaml



def window_sum(values, width, forward=True):
    """Truncated within-case sum, including the current hour; no edge repeats."""
    if not forward:
        return window_sum(values[:, ::-1], width, True)[:, ::-1]
    cumulative = np.pad(np.cumsum(values, axis=1, dtype=np.float64), ((0, 0), (1, 0)))
    left = np.arange(values.shape[1])
    right = np.minimum(left + width, values.shape[1])
    return cumulative[:, right] - cumulative[:, left]


def consecutive_true(mask):
    """Run length ending/starting at each position, reset at each case boundary."""
    past = np.zeros(mask.shape, dtype=np.float32)
    future = np.zeros_like(past)
    for h in range(mask.shape[1]):
        past[:, h] = mask[:, h] * (1 + (past[:, h - 1] if h else 0))
    for h in range(mask.shape[1] - 1, -1, -1):
        future[:, h] = mask[:, h] * (1 + (future[:, h + 1] if h + 1 < mask.shape[1] else 0))
    return past, future


def plant_sources(topology, plant):
    """Follow reservoir -> plant or reservoir -> tunnel -> plant connections.

    Stop at the first reservoir; do not pretend all upstream water arrives instantly.
    """
    connections = topology['connections']
    found = set()

    def visit(name, kind, seen):
        if (name, kind) in seen:
            return
        seen = seen | {(name, kind)}
        for edge in connections:
            if edge['to'] != name or edge['to_type'] != kind:
                continue
            if edge['from_type'] == 'reservoir':
                found.add(edge['from'])
            elif edge['from_type'] == 'tunnel':
                visit(edge['from'], 'tunnel', seen)

    visit(plant, 'plant', set())
    if not found:
        raise ValueError(f'No source reservoir found for {plant}')
    return sorted(found)


def build_research_features(data):
    """Return global additions, per-plant additions, and an auditable source mapping."""
    root = data.root
    with open(root / 'data/extended/Tokke_Vinje_topology.yaml') as stream:
        topology = yaml.safe_load(stream)
    starts = pd.DatetimeIndex(pd.concat([data.train_cases, data.test_cases])['starttime'])
    n = data.n_all_cases
    horizon = np.arange(HORIZON)
    time_index = starts.repeat(HORIZON) + pd.to_timedelta(np.tile(horizon, n), unit='h')
    price = data.X_all['price'].to_numpy().reshape(n, HORIZON)
    volume = read_time_csv(root / 'data/kernel/Historical_volume_2015_2024.csv').set_index('date').reindex(starts)
    water = read_time_csv(root / 'data/kernel/Synthetic_water_value_2015_2024.csv').set_index('date').reindex(starts + pd.Timedelta(days=7))
    inflow = read_time_csv(root / 'data/kernel/Historical_inflow_1958_2025.csv').set_index('date').reindex(time_index)
    minimum = read_time_csv(root / 'data/extended/Constraint_min_volume.csv').set_index('date').reindex(time_index)
    extra = {}

    def add(target, name, values):
        values = np.broadcast_to(values, (n, HORIZON)).reshape(-1).astype(np.float32)
        if not np.isfinite(values).all():
            raise ValueError(f'Non-finite research feature {name}')
        target[name] = values

    add(extra, 'v2_day_in_horizon', np.broadcast_to(horizon // 24, (n, HORIZON)))
    add(extra, 'v2_hours_remaining', HORIZON - horizon)
    day_price = price.reshape(n, 7, 24)
    day_mean = np.repeat(day_price.mean(axis=2), 24, axis=1)
    day_rank = pd.DataFrame(day_price.reshape(-1, 24)).rank(axis=1, pct=True).to_numpy().reshape(n, HORIZON)
    add(extra, 'v2_price_day_rank', day_rank)
    add(extra, 'v2_price_minus_day_mean', price - day_mean)
    prefix_mean = np.cumsum(price, axis=1) / (horizon + 1)
    suffix_mean = np.cumsum(price[:, ::-1], axis=1)[:, ::-1] / (HORIZON - horizon)
    add(extra, 'v2_price_minus_prefix_mean', price - prefix_mean)
    add(extra, 'v2_price_minus_suffix_mean', price - suffix_mean)
    add(extra, 'v2_best_remaining_price_gap', np.maximum.accumulate(price[:, ::-1], axis=1)[:, ::-1] - price)
    reservoirs = topology['model']['reservoir']
    reservoir_inputs = {}
    for name, config in reservoirs.items():
        cap = float(config['max_vol'])
        initial = volume[name].to_numpy()[:, None]
        terminal = water[name].to_numpy()[:, None]
        flow = inflow[name].to_numpy().reshape(n, HORIZON)
        # m3/s * 3600 / 1e6 -> million m3 per hour, matching storage units.
        hourly_volume = flow * 0.0036
        cumulative = np.cumsum(hourly_volume, axis=1)
        remaining = np.cumsum(hourly_volume[:, ::-1], axis=1)[:, ::-1]
        floor = minimum[name].to_numpy().reshape(n, HORIZON) if name in minimum else np.zeros_like(price)
        budget = initial + cumulative
        add(extra, f'v2_cumulative_inflow_fill__{name}', cumulative / cap)
        add(extra, f'v2_remaining_inflow_fill__{name}', remaining / cap)
        add(extra, f'v2_no_release_excess__{name}', np.maximum(budget - cap, 0) / cap)
        add(extra, f'v2_no_release_min_slack__{name}', (budget - floor) / cap)
        curve = config['vol_head']
        head = np.interp(initial[:, 0], curve['x'], curve['y'])[:, None]
        reservoir_inputs[name] = (cap, initial, terminal, cumulative, remaining, head)

    plants = {}
    mappings = []
    for plant, plant_config in topology['model']['plant'].items():
        sources = plant_sources(topology, plant)
        values = [reservoir_inputs[s] for s in sources]
        capacity = sum(v[0] for v in values)
        initial = sum(v[1] for v in values)
        # Multi-reservoir summaries are capacity-weighted approximations, not SHOP prices.
        terminal = sum(v[2] * v[0] for v in values) / capacity
        head = sum(v[5] * v[0] for v in values) / capacity - float(plant_config['outlet_line'])
        cum = sum(v[3] for v in values)
        remain = sum(v[4] for v in values)
        margin = price - terminal
        out = {}
        add(out, 'local_initial_fill', initial / capacity)
        add(out, 'local_gross_head_proxy', head)
        add(out, 'local_terminal_water_value', terminal)
        add(out, 'local_margin', margin)
        add(out, 'local_margin_normalized', margin / (np.abs(terminal) + price.std(axis=1)[:, None] + 1))
        add(out, 'local_cumulative_inflow_fill', cum / capacity)
        add(out, 'local_remaining_inflow_fill', remain / capacity)
        add(out, 'local_margin_x_fill', margin * initial / capacity)
        positive = np.maximum(margin, 0)
        negative = np.maximum(-margin, 0)
        past, future = consecutive_true(margin > 0)
        add(out, 'local_profitable_run_past', past)
        add(out, 'local_profitable_run_future', future)
        add(out, 'local_profitable_fraction_week', (margin > 0).mean(axis=1)[:, None])
        for hours in (6, 24):
            add(out, f'local_positive_margin_next_{hours}h', window_sum(positive, hours))
            add(out, f'local_negative_margin_next_{hours}h', window_sum(negative, hours))
            add(out, f'local_positive_margin_past_{hours}h', window_sum(positive, hours, False))
        # Start cost / rated production gives EUR/MWh; margin sums are hourly
        # EUR/MWh contributions. This is an approximate rated-power payback ratio.
        unit_configs = [topology['model']['generator'][u] for u in data.units if u.rsplit('_G', 1)[0] == plant]
        cost_per_mw = np.mean([(v.get('startcost_const', 0) + v.get('stopcost_const', 0)) / v['p_max'] for v in unit_configs])
        add(out, 'local_start_stop_payback_proxy', window_sum(positive, 24) / max(cost_per_mw, 1e-6))
        plants[plant] = pd.DataFrame(out)
        mappings.append({'plant': plant, 'source_reservoirs': '|'.join(sources), 'source_capacity_Mm3': capacity})
    return pd.DataFrame(extra), plants, pd.DataFrame(mappings)

## fit_helpers — definitions only

In [5]:
"""Sequential, resumable challenger. Importing this module never trains models."""
import gc
import hashlib
import importlib.metadata
import json
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from threadpoolctl import threadpool_limits


def raw_digest(root):
    paths = sorted((root / 'data').rglob('*.csv')) + sorted((root / 'data').rglob('*.yaml'))
    paths += [root / 'prediction_mapping.csv', root / 'sample_submission.csv']
    digest = hashlib.sha256()
    for path in paths:
        digest.update(str(path.relative_to(root)).encode())
        with path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
    return digest.hexdigest()


def save_json(path, value):
    path = Path(path)
    temporary = path.with_suffix('.tmp.json')
    temporary.write_text(json.dumps(value, indent=2), encoding='utf-8')
    os.replace(temporary, path)


def purged_year(data, year):
    starts = pd.DatetimeIndex(data.train_cases['starttime'])
    begin = pd.Timestamp(f'{year}-01-01', tz='UTC')
    end = pd.Timestamp(f'{year+1}-01-01', tz='UTC')
    train = np.flatnonzero(starts + pd.Timedelta(hours=HORIZON-1) < begin)
    valid = np.flatnonzero((starts >= begin) & (starts < end))
    if not len(train) or not len(valid):
        raise ValueError('Empty temporal split')
    assert starts[train].max() + pd.Timedelta(hours=HORIZON-1) < starts[valid].min()
    return train, valid


def score_cube(truth, prediction):
    return float(roc_auc_score(truth.ravel(), prediction.ravel()))


def fit_member(spec, X, y, rounds, threads, valid=None):
    parameters = dict(spec['parameters'])
    if spec['backend'] == 'xgboost':
        from xgboost import XGBClassifier
        model = XGBClassifier(**parameters, n_estimators=int(rounds), n_jobs=threads,
                              **({'early_stopping_rounds': 80} if valid is not None else {}))
        model.fit(X, y, eval_set=[valid] if valid is not None else None, verbose=False)
        best = int(model.best_iteration) + 1 if valid is not None else int(rounds)
    else:
        import lightgbm as lgb
        parameters['num_threads'] = threads
        training = lgb.Dataset(X, label=y, free_raw_data=True)
        options = {}
        if valid is not None:
            parameters['metric'] = 'auc'
            options = dict(valid_sets=[lgb.Dataset(valid[0], label=valid[1], reference=training)],
                           callbacks=[lgb.early_stopping(80, verbose=False)])
        model = lgb.train(parameters, training, num_boost_round=int(rounds), **options)
        best = int(model.best_iteration or rounds)
    return model, best

## competition — definitions only

In [6]:
"""Feature-only refinements for the frozen notebook 12/13 estimators."""
import numpy as np
import pandas as pd
import yaml


def price_competition(price, widths=(24, 72, 168)):
    """Strictly better alternative hours within this case, excluding current hour.

    Compute relative position and mean forgone price premium separately before
    and after the current hour. No edge replication or cross-case windows.
    """
    price = np.asarray(price, dtype=np.float64)
    n, horizon = price.shape
    ahead_count = np.zeros_like(price); past_count = np.zeros_like(price)
    ahead_gain = np.zeros_like(price); past_gain = np.zeros_like(price)
    scale = price.std(axis=1)[:, None] + 1.
    output = {}
    for distance in range(1, max(widths)):
        if distance < horizon:
            gap = price[:, distance:] - price[:, :-distance]
            ahead_count[:, :-distance] += gap > 0
            past_count[:, distance:] += gap < 0
            ahead_gain[:, :-distance] += np.maximum(gap, 0)
            past_gain[:, distance:] += np.maximum(-gap, 0)
        width = distance + 1
        if width in widths:
            ahead_n = np.maximum(1, np.minimum(width-1, horizon-1-np.arange(horizon)))
            past_n = np.maximum(1, np.minimum(width-1, np.arange(horizon)))
            output[f'v4_better_future_fraction_{width}'] = ahead_count / ahead_n
            output[f'v4_better_past_fraction_{width}'] = past_count / past_n
            output[f'v4_future_premium_{width}'] = ahead_gain / ahead_n / scale
            output[f'v4_past_premium_{width}'] = past_gain / past_n / scale
    return {k: v.astype(np.float32) for k, v in output.items()}


def build_refinement_features(data):
    n = data.n_all_cases
    price = data.X_all['price'].to_numpy().reshape(n, HORIZON)
    common = pd.DataFrame({k: v.ravel() for k, v in price_competition(price).items()})
    starts = pd.DatetimeIndex(pd.concat([data.train_cases, data.test_cases])['starttime'])
    times = starts.repeat(HORIZON) + pd.to_timedelta(np.tile(np.arange(HORIZON), n), unit='h')
    with open(data.root / 'data/extended/Tokke_Vinje_topology.yaml') as stream:
        topology = yaml.safe_load(stream)
    volumes = read_time_csv(data.root / 'data/kernel/Historical_volume_2015_2024.csv').set_index('date').reindex(starts)
    water = read_time_csv(data.root / 'data/kernel/Synthetic_water_value_2015_2024.csv').set_index('date').reindex(starts + pd.Timedelta(days=7))
    inflow = read_time_csv(data.root / 'data/kernel/Historical_inflow_1958_2025.csv').set_index('date').reindex(times)
    minimum = read_time_csv(data.root / 'data/extended/Constraint_min_volume.csv').set_index('date').reindex(times)
    price_only, combined, audit = {}, {}, []
    for plant in topology['model']['plant']:
        sources = plant_sources(topology, plant)
        cap = sum(topology['model']['reservoir'][s]['max_vol'] for s in sources)
        terminal = sum(water[s].to_numpy()[:, None] * topology['model']['reservoir'][s]['max_vol'] for s in sources) / cap
        margin = price - terminal
        flow = inflow[sources].sum(axis=1).to_numpy().reshape(n, HORIZON)
        initial = volumes[sources].sum(axis=1).to_numpy()[:, None]
        floor = sum((minimum[s].to_numpy().reshape(n, HORIZON) for s in sources if s in minimum), np.zeros_like(price))
        out = {}

        def add(name, values):
            values = np.broadcast_to(values, (n, HORIZON)).ravel().astype(np.float32)
            if not np.isfinite(values).all():
                raise ValueError(f'Non-finite feature {plant}/{name}')
            out['v4_' + name] = values

        prefix = np.arange(1, HORIZON+1)
        suffix = prefix[::-1]
        positive = margin > 0
        add('profitable_prefix_fraction', np.cumsum(positive, axis=1) / prefix)
        add('profitable_suffix_fraction', np.cumsum(positive[:, ::-1], axis=1)[:, ::-1] / suffix)
        inflow_cumulative = np.cumsum(flow, axis=1)
        flow_total = flow.sum(axis=1)[:, None]
        fraction_arrived = np.divide(inflow_cumulative, flow_total,
                                     out=np.zeros_like(inflow_cumulative), where=flow_total > 0)
        profit = np.maximum(margin, 0)
        profit_total = profit.sum(axis=1)[:, None]
        fraction_profit_elapsed = np.divide(np.cumsum(profit, axis=1), profit_total,
                                            out=np.zeros_like(profit), where=profit_total > 0)
        add('inflow_arrival_fraction', fraction_arrived)
        add('profit_elapsed_fraction', fraction_profit_elapsed)
        add('inflow_minus_profit_timing', fraction_arrived - fraction_profit_elapsed)
        add('no_profitable_hours', profit_total == 0)
        add('no_inflow', flow_total == 0)
        for width in (24, 72, 168):
            flow_window = window_sum(flow, width)
            weighted = np.divide(window_sum(flow*price, width), flow_window,
                                 out=np.zeros_like(price, dtype=np.float64), where=flow_window > 0)
            add(f'inflow_weighted_price_gap_{width}', np.where(flow_window > 0, weighted-price, 0))
            add(f'profitable_inflow_fill_{width}', window_sum(flow*positive, width)*.0036/cap)
        future_floor = np.maximum.accumulate(floor[:, ::-1], axis=1)[:, ::-1]
        add('upcoming_floor_increase_fill', (future_floor-floor)/cap)
        add('initial_slack_to_future_floor', (initial-future_floor)/cap)
        add('hours_until_highest_remaining_price', _hours_to_future_max(price))
        add('profitable_hours_remaining', np.cumsum(positive[:, ::-1], axis=1)[:, ::-1])
        water_features = pd.DataFrame(out)
        price_only[plant] = common
        combined[plant] = pd.concat([common, water_features], axis=1)
        audit.append(dict(plant=plant, sources='|'.join(sources),
                          price_features=len(common.columns), timing_features=len(water_features.columns)))
    return {'competition': price_only, 'competition_timing': combined}, pd.DataFrame(audit)


def _hours_to_future_max(price):
    n, horizon = price.shape
    result = np.zeros_like(price)
    best = np.full(n, -np.inf)
    index = np.full(n, horizon-1)
    for t in range(horizon-1, -1, -1):
        update = price[:, t] >= best
        best = np.maximum(best, price[:, t])
        index = np.where(update, t, index)
        result[:, t] = index-t
    return result

## paths — definitions only

In [7]:
"""Fixed, input-only whole-week descriptors for notebook 18.

No labels, fitted transforms, future observed storage, or cross-case windows.
The supplied future price/inflow paths are inputs to each scheduling problem.
"""
import numpy as np
import pandas as pd
import yaml
from scipy.fft import dct


def contiguous_competition(price):
    """Consecutive strictly better hours immediately before/after each hour.

    Unlike a rank, an interruption ends the sequence. Current hour is excluded.
    Equal prices end a run. Each case is independent.
    """
    price = np.asarray(price)
    n, h = price.shape
    result = {}
    for label, values in [('next', price), ('past', price[:, ::-1])]:
        active = np.ones((n, h), dtype=bool)
        count = np.zeros((n, h), np.float32)
        premium = np.zeros((n, h), np.float32)
        for shift in range(1, h):
            gap = values[:, shift:] - values[:, :-shift]
            active[:, :-shift] &= gap > 0
            count[:, :-shift] += active[:, :-shift]
            premium[:, :-shift] += np.maximum(gap, 0) * active[:, :-shift]
        result[f'path_consecutive_better_{label}'] = count if label=='next' else count[:, ::-1]
        result[f'path_contiguous_premium_{label}'] = premium if label=='next' else premium[:, ::-1]
    return result


def weekly_shape(price):
    """Daily and low-frequency shape, preserving absolute day positions in a case."""
    price = np.asarray(price, dtype=np.float64)
    n, h = price.shape
    if h != HORIZON:
        raise ValueError('Expected exactly 168 hourly prices')
    scale = price.std(axis=1)[:, None] + 1.
    centered = (price-price.mean(axis=1)[:, None])/scale
    out = {}
    daily = price.reshape(n, 7, 24)
    for day in range(7):
        # Compare current hour with EACH day, not just its own day or whole week.
        out[f'path_day{day}_mean_gap'] = (daily[:, day].mean(axis=1)[:, None]-price)/scale
        out[f'path_day{day}_peak_gap'] = (daily[:, day].max(axis=1)[:, None]-price)/scale
    coefficients = dct(centered, type=2, norm='ortho', axis=1)
    for k in range(1, 7):
        out[f'path_price_dct_{k}'] = np.broadcast_to(coefficients[:, k, None], (n,h))
    for forward, label in [(True,'next'), (False,'past')]:
        for width in (48, 96, 168):
            counts = np.minimum(width, np.arange(h,0,-1) if forward else np.arange(1,h+1))
            average = window_sum(price,width,forward)/counts
            out[f'path_price_{label}_{width}_relative'] = (price-average)/scale
    for name, values in contiguous_competition(price).items():
        out[name] = values/scale if 'premium' in name else values
    return {name: np.asarray(value, dtype=np.float32) for name,value in out.items()}


def build_path_features(data, local):
    n = data.n_all_cases
    price = data.X_all['price'].to_numpy().reshape(n,HORIZON)
    global_frame = pd.DataFrame({k:v.ravel() for k,v in weekly_shape(price).items()})
    with open(data.root/'data/extended/Tokke_Vinje_topology.yaml') as stream:
        topology = yaml.safe_load(stream)
    frames, records = {}, []
    for plant in local:
        sources = plant_sources(topology,plant)
        cap = sum(topology['model']['reservoir'][r]['max_vol'] for r in sources)
        terminal = local[plant]['local_terminal_water_value'].to_numpy().reshape(n,HORIZON)[:,0,None]
        margin = price-terminal
        scale = price.std(axis=1)[:,None]+np.abs(terminal)+1.
        daily_margin = margin.reshape(n,7,24)
        # All inputs come from X_all, already restricted to the supplied case window.
        flow = sum(data.X_all[f'inflow__{r}'].to_numpy().reshape(n,HORIZON) for r in sources)
        daily_flow = flow.reshape(n,7,24)
        extra = {}
        def add(name, value):
            extra['path_'+name] = np.broadcast_to(value,(n,HORIZON)).ravel().astype(np.float32)
        for day in range(7):
            add(f'day{day}_positive_margin', np.maximum(daily_margin[:,day],0).sum(axis=1)[:,None]/scale)
            add(f'day{day}_negative_margin', np.maximum(-daily_margin[:,day],0).sum(axis=1)[:,None]/scale)
            add(f'day{day}_source_inflow_fill',daily_flow[:,day].sum(axis=1)[:,None]*.0036/cap)
        profit=np.maximum(margin,0)
        profit_sum=profit.sum(axis=1)[:,None]
        shares=np.divide(profit,profit_sum,out=np.zeros_like(profit),where=profit_sum>0)
        add('profit_concentration', (shares**2).sum(axis=1)[:,None])
        add('profit_center_of_time', (shares*np.arange(HORIZON)).sum(axis=1)[:,None]/(HORIZON-1))
        add('current_profit_share',shares)
        frame=pd.concat([global_frame,pd.DataFrame(extra)],axis=1)
        if not np.isfinite(frame.to_numpy()).all() or not frame.columns.is_unique:
            raise ValueError(f'Invalid path features for {plant}')
        frames[plant]=frame
        records.append(dict(plant=plant,sources='|'.join(sources),global_features=len(global_frame.columns),local_features=len(extra)))
    return frames,pd.DataFrame(records)

## runner — definitions only

In [8]:
"""Bounded staged boosting experiment. Importing this module never trains."""
import gc
import hashlib
import importlib.metadata
import json
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from threadpoolctl import threadpool_limits


def factor_key(factor):
    return f'{float(factor):g}'


def scaled_rounds(original, factor):
    if not 0 < float(factor) <= 2:
        raise ValueError('Round multiplier must be in (0, 2]')
    return max(1, int(round(int(original)*float(factor))))


def checked_cube(values, cases, units):
    values=np.asarray(values,dtype=np.float32)
    if values.shape != (cases,units,HORIZON) or not np.isfinite(values).all():
        raise ValueError('Invalid prediction dimensions or non-finite values')
    if values.min()<0 or values.max()>1:
        raise ValueError('Prediction outside [0,1]')
    return values


def staged_predict(model, backend, X, rounds, threads):
    if backend=='xgboost':
        return model.predict_proba(X,iteration_range=(0,int(rounds)))[:,1]
    return model.predict(X,num_iteration=int(rounds),num_threads=threads)


class PathRunner:
    def __init__(self,data,extra,local,competition,paths,specs,rounds,threads,
                 source_digest,v4_source_digest,use_checkpoints=True):
        self.data,self.local,self.competition,self.paths=data,local,competition,paths
        self.specs,self.rounds=specs,rounds
        self.threads=max(1,min(int(threads),os.cpu_count() or 1))
        self.use_checkpoints=bool(use_checkpoints)
        self.source_digest,self.v4_source_digest=source_digest,v4_source_digest
        self.base_names=list(data.feature_sets['full_engineered'])+list(extra.columns)
        self.base=np.column_stack([data.X_all[data.feature_sets['full_engineered']].to_numpy(np.float32),extra.to_numpy(np.float32)])
        self.raw=raw_digest(data.root)
        self.directory=data.root/'artifacts/multiscale_boosting_v5'
        self.directory.mkdir(parents=True,exist_ok=True)

    def identity(self):
        return dict(source=self.source_digest,raw=self.raw,specs=self.specs,rounds=self.rounds,
                    packages={p:importlib.metadata.version(p) for p in ('numpy','pandas','scikit-learn','xgboost','lightgbm')})

    def _legacy_cube(self,family,train,predict):
        """Reuse ONLY exact v4-competition runs; no guessing from filenames alone."""
        if not self.use_checkpoints:
            return None
        member=family+'_competition'
        for path in sorted((self.data.root/'artifacts/physics_features_v4'/member).glob('*/manifest.json')):
            m=json.loads(path.read_text())
            spec=self.specs[family]
            wanted_packages={p:importlib.metadata.version(p) for p in ('numpy','pandas','scikit-learn',spec['backend'])}
            if not (m.get('raw')==self.raw and m.get('source')==self.v4_source_digest
                    and m.get('member')==member and m.get('spec')==spec
                    and m.get('train')==list(map(int,train)) and m.get('predict')==list(map(int,predict))
                    and m.get('rounds')==self.rounds[family] and not m.get('tune')
                    and m.get('packages')==wanted_packages):
                continue
            pieces=[]
            for i,unit in enumerate(self.data.units):
                checkpoint=path.parent/f'unit_{i:02d}.npz'
                if not checkpoint.exists():
                    break
                with np.load(checkpoint,allow_pickle=False) as saved:
                    if int(saved['rounds']) != self.rounds[family][unit]:
                        break
                    pieces.append(saved['prediction'].copy())
            if len(pieces)==len(self.data.units):
                cube=checked_cube(np.stack(pieces,axis=1),len(predict),len(self.data.units))
                save_json(self.directory/f'imported_{family}_{int(predict[0])}.json',dict(
                    path=str(path.relative_to(self.data.root)),sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
                    source=m['source'],raw=self.raw,note='Matching notebook-17 competition predictions; thread counts may differ.'))
                print(f'Restored exact v4 {family}: {path.parent}',flush=True)
                return cube
        return None

    def run(self,family,variant,train,predict,stage,factors=(1.,1.5,2.)):
        if family not in self.specs or variant not in ('v4','path'):
            raise ValueError('Unknown model or feature variant')
        train,predict=np.asarray(train,dtype=int),np.asarray(predict,dtype=int)
        if not len(train) or not len(predict) or len(np.intersect1d(train,predict)):
            raise ValueError('Empty or overlapping training/prediction cases')
        if np.any(train<0) or np.any(train>=self.data.n_train_cases) or np.any(predict<0) or np.any(predict>=self.data.n_all_cases):
            raise ValueError('Case index outside dataset')
        factors=tuple(sorted(set(float(f) for f in factors)))
        if not factors:
            raise ValueError('No requested tree counts')
        for f in factors: scaled_rounds(1,f)
        if variant=='v4' and factors==(1.,):
            restored=self._legacy_cube(family,train,predict)
            if restored is not None:
                return {'1':restored}
        manifest=dict(self.identity(),family=family,variant=variant,stage=stage,threads=self.threads,
                      train=list(map(int,train)),predict=list(map(int,predict)),factors=list(factors))
        signature=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()[:24]
        directory=self.directory/f'{stage}_{family}_{variant}'/signature
        directory.mkdir(parents=True,exist_ok=True)
        save_json(directory/'manifest.json',manifest)
        cubes={factor_key(f):np.empty((len(predict),len(self.data.units),HORIZON),np.float32) for f in factors}
        train_rows,pred_rows=rows_for_cases(train),rows_for_cases(predict)
        timings=[]
        for i,unit in enumerate(self.data.units):
            start=time.perf_counter()
            counts=[scaled_rounds(self.rounds[family][unit],f) for f in factors]
            checkpoint=directory/f'unit_{i:02d}.npz'
            if self.use_checkpoints and checkpoint.exists():
                with np.load(checkpoint,allow_pickle=False) as saved:
                    if saved['rounds'].tolist()!=counts:
                        raise ValueError(f'Mismatched checkpoint: {checkpoint}')
                    predictions=saved['predictions'].copy()
                status='restored'
            else:
                plant=unit.rsplit('_G',1)[0]
                parts=[self.base,self.local[plant].to_numpy(np.float32),self.competition[plant].to_numpy(np.float32)]
                names=self.base_names+list(self.local[plant].columns)+list(self.competition[plant].columns)
                if variant=='path':
                    parts.append(self.paths[plant].to_numpy(np.float32)); names+=list(self.paths[plant].columns)
                if len(set(names))!=len(names):
                    raise ValueError('Duplicate model features')
                X=np.ascontiguousarray(np.column_stack(parts),dtype=np.float32)
                del parts
                y=self.data.target_cube[train,i].ravel()
                print(f'{stage} | {family}/{variant} | {unit} | {max(counts)} trees; snapshots {counts}',flush=True)
                if len(np.unique(y))==1:
                    predictions=np.full((len(factors),len(predict),HORIZON),float(y[0]),np.float32)
                else:
                    with threadpool_limits(limits=self.threads),threadpool_limits(limits=1,user_api='blas'):
                        model,_=fit_member(self.specs[family],X[train_rows],y,max(counts),self.threads)
                        pred_X=X[pred_rows]
                        # One fit, three prefixes. No early stopping on the audit labels.
                        predictions=np.stack([staged_predict(model,self.specs[family]['backend'],pred_X,count,self.threads).reshape(len(predict),HORIZON) for count in counts]).astype(np.float32)
                        gain=model.feature_importances_ if self.specs[family]['backend']=='xgboost' else model.feature_importance(importance_type='gain')
                    pd.DataFrame(dict(feature=names,gain=gain)).sort_values('gain',ascending=False).to_csv(directory/f'importance_{i:02d}.csv',index=False)
                    del model,pred_X
                temporary=checkpoint.with_suffix('.tmp.npz')
                np.savez_compressed(temporary,predictions=predictions,rounds=np.asarray(counts))
                os.replace(temporary,checkpoint)
                del X,y
                gc.collect()
                status='trained'
            if predictions.shape!=(len(factors),len(predict),HORIZON) or not np.isfinite(predictions).all() or predictions.min()<0 or predictions.max()>1:
                raise ValueError(f'Invalid checkpoint: {checkpoint}')
            for k,f in enumerate(factors): cubes[factor_key(f)][:,i]=predictions[k]
            timings.append(dict(unit=unit,max_rounds=max(counts),seconds=time.perf_counter()-start,status=status))
            pd.DataFrame(timings).to_csv(directory/'runtime.csv',index=False)
            print(f'  {status}: {timings[-1]["seconds"]:.1f}s ({i+1}/{len(self.data.units)})',flush=True)
        return cubes


def select_pair(scores,predictions,truth):
    """Small, global search on development only; ties prefer existing features/counts."""
    chosen={}
    for family in ('xgb','lgb'):
        candidates=[row for row in scores if row['family']==family]
        best=max(candidates,key=lambda r:(r['micro_auc'],r['variant']=='v4',-r['factor']))
        chosen[family]=dict(variant=best['variant'],factor=best['factor'])
    x=predictions[('xgb',chosen['xgb']['variant'],factor_key(chosen['xgb']['factor']))]
    l=predictions[('lgb',chosen['lgb']['variant'],factor_key(chosen['lgb']['factor']))]
    grid=[dict(xgb_weight=w,micro_auc=score_cube(truth,w*x+(1-w)*l)) for w in (.25,.5,.75)]
    best=max(grid,key=lambda r:(r['micro_auc'],-abs(r['xgb_weight']-.5)))
    return dict(members=chosen,xgb_weight=best['xgb_weight'],development_auc=best['micro_auc']),grid


def develop_paths(runner):
    train,valid=purged_year(runner.data,2021)
    predictions,rows={},[]
    for family in ('xgb','lgb'):
        for variant in ('v4','path'):
            snapshots=runner.run(family,variant,train,valid,'development_2021')
            for factor,p in snapshots.items():
                predictions[(family,variant,factor)]=p
                row=dict(year=2021,family=family,variant=variant,factor=float(factor),micro_auc=score_cube(runner.data.target_cube[valid],p))
                rows.append(row)
                print(row,flush=True)
                pd.DataFrame(rows).to_csv(runner.directory/'development_scores.csv',index=False)
    selection,grid=select_pair(rows,predictions,runner.data.target_cube[valid])
    selection['identity']=runner.identity()
    selection['development_baseline_auc']=score_cube(runner.data.target_cube[valid],.5*(predictions[('xgb','v4','1')]+predictions[('lgb','v4','1')]))
    save_json(runner.directory/'selection.json',selection)
    pd.DataFrame(grid).to_csv(runner.directory/'blend_grid_2021.csv',index=False)
    return selection,pd.DataFrame(rows)


def diagnostic_tables(data,cases,predictions):
    """Descriptive per-unit/month/horizon errors; never used for unit-specific tuning."""
    rows=[]
    truth=data.target_cube[cases]
    months=pd.DatetimeIndex(data.train_cases.iloc[cases].starttime).month
    for name,p in predictions.items():
        for group,labels in [('unit',list(data.units)),('month',list(range(1,13))),('horizon_day',list(range(1,8)))]:
            for i,label in enumerate(labels):
                if group=='unit': y,q=truth[:,i],p[:,i]
                elif group=='month': y,q=truth[months==label],p[months==label]
                else: y,q=truth[:,:,(label-1)*24:label*24],p[:,:,(label-1)*24:label*24]
                if y.size:
                    auc=float(roc_auc_score(y.ravel(),q.ravel())) if len(np.unique(y))==2 else np.nan
                    rows.append(dict(model=name,group=group,label=label,auc=auc,brier=float(np.mean((q-y)**2)),on_rate=float(y.mean()),predicted_rate=float(q.mean())))
    return pd.DataFrame(rows)


def audit_paths(runner,selection):
    if selection.get('identity')!=runner.identity():
        raise ValueError('Selection configuration changed; rerun development')
    train,valid=purged_year(runner.data,2022)
    chosen,baseline={},{}
    for family in ('xgb','lgb'):
        member=selection['members'][family]
        if member['variant']=='v4':
            snapshots=runner.run(family,'v4',train,valid,'audit_2022',factors=(1.,member['factor']))
            baseline[family]=snapshots['1'];chosen[family]=snapshots[factor_key(member['factor'])]
        else:
            baseline[family]=runner.run(family,'v4',train,valid,'audit_2022',factors=(1.,))['1']
            chosen[family]=runner.run(family,'path',train,valid,'audit_2022',factors=(member['factor'],))[factor_key(member['factor'])]
    weight=selection['xgb_weight']
    candidate=weight*chosen['xgb']+(1-weight)*chosen['lgb']
    control=.5*(baseline['xgb']+baseline['lgb'])
    new_score,old_score=score_cube(runner.data.target_cube[valid],candidate),score_cube(runner.data.target_cube[valid],control)
    selection=dict(selection,audit_auc=new_score,audit_baseline_auc=old_score,audit_gain=new_score-old_score,passed_audit=bool(new_score>old_score))
    save_json(runner.directory/'selection.json',selection)
    np.savez_compressed(runner.directory/'audit_predictions.npz',case_ids=valid,candidate=candidate,baseline=control)
    diagnostic_tables(runner.data,valid,dict(candidate=candidate,baseline=control)).to_csv(runner.directory/'audit_diagnostics.csv',index=False)
    print(f'2022: candidate={new_score:.6f}; v4 baseline={old_score:.6f}; delta={new_score-old_score:+.6f}',flush=True)
    print('Historical audit improved.' if selection['passed_audit'] else 'NO confirmed improvement. Keep the existing Kaggle-best CSV as a fallback.',flush=True)
    return selection


def final_paths(runner,selection,filename='multiscale_boosting_v5.csv'):
    if selection.get('identity')!=runner.identity():
        raise ValueError('Selection configuration/source/data/packages changed')
    train=np.arange(runner.data.n_train_cases)
    test=np.arange(runner.data.n_train_cases,runner.data.n_all_cases)
    blend=np.zeros((len(test),len(runner.data.units),HORIZON),np.float32)
    for family in ('xgb','lgb'):
        member=selection['members'][family]
        p=runner.run(family,member['variant'],train,test,'final',factors=(member['factor'],))[factor_key(member['factor'])]
        weight=selection['xgb_weight'] if family=='xgb' else 1-selection['xgb_weight']
        blend+=weight*p
        path,_=make_submission(runner.data,p,f'{family}_multiscale_v5.csv')
        print('Single model:',path,flush=True)
    path,submission=make_submission(runner.data,checked_cube(blend,len(test),len(runner.data.units)),filename)
    print('SUBMISSION READY:',path,flush=True)
    return path,submission

## Build features — no training

In [9]:
# Input-only feature construction. No estimators are fitted in this cell.
data=build_project_data(ROOT)
extra,local,mapping=build_research_features(data)
additions,competition_audit=build_refinement_features(data)
competition=additions['competition']
del additions  # Do not retain the rejected competition+timing variant.
paths,path_audit=build_path_features(data,local)
runner=PathRunner(data,extra,local,competition,paths,SPECS,ROUNDS,THREADS,
                  SOURCE_DIGEST,V4_SOURCE_DIGEST,USE_CHECKPOINTS)
path_audit.to_csv(runner.directory/'feature_manifest.csv',index=False)
display(path_audit)
print('Training cases:',data.n_train_cases,'test cases:',data.n_test_cases)
print('New path-feature arrays (MB):',round(sum(f.memory_usage().sum() for f in paths.values())/1e6,1))
print('Research: 56 development unit fits, <=56 audit fits, 28 final fits.')
print('Each development fit provides three tree-count snapshots; not three separate trainings.')

,plant,sources,global_features,local_features
0,Hogga,Bandak,30,24
1,Tokke,Vinjevatn,30,24
2,Lio,Byrtevatn,30,24
3,Byrte,Botnedalsvatn,30,24
4,Vinje,Vaamarvatn,30,24
5,Haukeli,Vatjern,30,24
6,Songa,Bitdalsvatn|Songavatn,30,24
7,Kjela,Foersvatn,30,24
8,Vesle Kjela,Kjelavatn,30,24


Training cases: 2922 test cases: 725
New path-feature arrays (MB): 1191.1
Research: 56 development unit fits, <=56 audit fits, 28 final fits.
Each development fit provides three tree-count snapshots; not three separate trainings.


## TRAINING A — controlled development

In [10]:
# TRAINING A — select features, round multiplier, and one global blend weight on 2021.
selection,scores=develop_paths(runner)
display(scores.sort_values('micro_auc',ascending=False))
print('Chosen configuration:',selection['members'],'XGB weight:',selection['xgb_weight'])
print('Development blend:',selection['development_auc'],'v4 control:',selection['development_baseline_auc'])

development_2021 | xgb/v4 | Hogga_G1 | 820 trees; snapshots [410, 615, 820]
  trained: 25.8s (1/14)
development_2021 | xgb/v4 | Tokke_G1 | 900 trees; snapshots [450, 675, 900]
  trained: 33.1s (2/14)
development_2021 | xgb/v4 | Tokke_G2 | 1708 trees; snapshots [854, 1281, 1708]
  trained: 54.7s (3/14)
development_2021 | xgb/v4 | Tokke_G3 | 1174 trees; snapshots [587, 880, 1174]
  trained: 40.3s (4/14)
development_2021 | xgb/v4 | Tokke_G4 | 1386 trees; snapshots [693, 1040, 1386]
  trained: 44.1s (5/14)
development_2021 | xgb/v4 | Lio_G1 | 498 trees; snapshots [249, 374, 498]
  trained: 16.6s (6/14)
development_2021 | xgb/v4 | Byrte_G1 | 584 trees; snapshots [292, 438, 584]
  trained: 19.2s (7/14)
development_2021 | xgb/v4 | Vinje_G1 | 1316 trees; snapshots [658, 987, 1316]
  trained: 43.2s (8/14)
development_2021 | xgb/v4 | Vinje_G2 | 808 trees; snapshots [404, 606, 808]
  trained: 27.5s (9/14)
development_2021 | xgb/v4 | Vinje_G3 | 1214 trees; snapshots [607, 910, 1214]
  trained: 38.

,year,family,variant,factor,micro_auc
11,2021,lgb,path,2.0,0.984838
5,2021,xgb,path,2.0,0.984750
10,2021,lgb,path,1.5,0.984738
4,2021,xgb,path,1.5,0.984685
3,2021,xgb,path,1.0,0.984467
9,2021,lgb,path,1.0,0.984376
2,2021,xgb,v4,2.0,0.984301
1,2021,xgb,v4,1.5,0.984277
0,2021,xgb,v4,1.0,0.984193
7,2021,lgb,v4,1.5,0.984033


Chosen configuration: {'xgb': {'variant': 'path', 'factor': 2.0}, 'lgb': {'variant': 'path', 'factor': 2.0}} XGB weight: 0.5
Development blend: 0.9851445788051693 v4 control: 0.984532444500612


## TRAINING B — frozen historical audit

In [11]:
# TRAINING B — frozen 2022 historical audit; no retuning against these labels.
# Restart safely: run resource/config/definition/build cells, then this cell.
selection=json.loads((runner.directory/'selection.json').read_text())
selection=audit_paths(runner,selection)
display(pd.read_csv(runner.directory/'audit_diagnostics.csv').query("group == 'unit'"))

Restored exact v4 xgb: /Users/henrik/Documents/H26/Moderne Maksinlæring/sintef_project/artifacts/physics_features_v4/xgb_competition/5d0d1dd60974280a68d3
audit_2022 | xgb/path | Hogga_G1 | 820 trees; snapshots [820]
  trained: 34.4s (1/14)
audit_2022 | xgb/path | Tokke_G1 | 900 trees; snapshots [900]
  trained: 44.6s (2/14)
audit_2022 | xgb/path | Tokke_G2 | 1708 trees; snapshots [1708]
  trained: 72.5s (3/14)
audit_2022 | xgb/path | Tokke_G3 | 1174 trees; snapshots [1174]
  trained: 53.6s (4/14)
audit_2022 | xgb/path | Tokke_G4 | 1386 trees; snapshots [1386]
  trained: 59.7s (5/14)
audit_2022 | xgb/path | Lio_G1 | 498 trees; snapshots [498]
  trained: 22.0s (6/14)
audit_2022 | xgb/path | Byrte_G1 | 584 trees; snapshots [584]
  trained: 25.2s (7/14)
audit_2022 | xgb/path | Vinje_G1 | 1316 trees; snapshots [1316]
  trained: 57.0s (8/14)
audit_2022 | xgb/path | Vinje_G2 | 808 trees; snapshots [808]
  trained: 36.6s (9/14)
audit_2022 | xgb/path | Vinje_G3 | 1214 trees; snapshots [1214]
  

,model,group,label,auc,brier,on_rate,predicted_rate
0,candidate,unit,Hogga_G1,0.978844,0.058082,0.386595,0.410165
1,candidate,unit,Tokke_G1,0.994766,0.029245,0.456670,0.455127
2,candidate,unit,Tokke_G2,0.988504,0.046422,0.357127,0.348354
3,candidate,unit,Tokke_G3,0.993002,0.034506,0.410877,0.401017
4,candidate,unit,Tokke_G4,0.988254,0.045641,0.373255,0.356676
5,candidate,unit,Lio_G1,0.962559,0.078217,0.549918,0.547580
6,candidate,unit,Byrte_G1,0.998718,0.015369,0.383089,0.375099
7,candidate,unit,Vinje_G1,0.986367,0.046253,0.343542,0.334528
8,candidate,unit,Vinje_G2,0.982011,0.049710,0.296999,0.287970
9,candidate,unit,Vinje_G3,0.974402,0.053007,0.192042,0.206450


## TRAINING C — submission files

In [12]:
# TRAINING C — selected pair on all labeled cases, then actual submission CSVs.
selection=json.loads((runner.directory/'selection.json').read_text())
if 'audit_auc' not in selection:
    raise ValueError('Run TRAINING B before final training')
if not selection['passed_audit']:
    print('WARNING: candidate did not beat v4 on 2022; this export is experimental, not a proven upgrade.')
submission_path,submission=final_paths(runner,selection)
display(submission.iloc[:3,:8])
print('Total minutes:',round((time.perf_counter()-NOTEBOOK_STARTED)/60,1))

final | xgb/path | Hogga_G1 | 820 trees; snapshots [820]
  trained: 38.6s (1/14)
final | xgb/path | Tokke_G1 | 900 trees; snapshots [900]
  trained: 50.7s (2/14)
final | xgb/path | Tokke_G2 | 1708 trees; snapshots [1708]
  trained: 81.9s (3/14)
final | xgb/path | Tokke_G3 | 1174 trees; snapshots [1174]
  trained: 60.4s (4/14)
final | xgb/path | Tokke_G4 | 1386 trees; snapshots [1386]
  trained: 66.8s (5/14)
final | xgb/path | Lio_G1 | 498 trees; snapshots [498]
  trained: 25.7s (6/14)
final | xgb/path | Byrte_G1 | 584 trees; snapshots [584]
  trained: 27.8s (7/14)
final | xgb/path | Vinje_G1 | 1316 trees; snapshots [1316]
  trained: 63.8s (8/14)
final | xgb/path | Vinje_G2 | 808 trees; snapshots [808]
  trained: 40.5s (9/14)
final | xgb/path | Vinje_G3 | 1214 trees; snapshots [1214]
  trained: 57.8s (10/14)
final | xgb/path | Haukeli_G1 | 446 trees; snapshots [446]
  trained: 23.1s (11/14)
final | xgb/path | Songa_G1 | 1020 trees; snapshots [1020]
  trained: 52.5s (12/14)
final | xgb/p

,Run No,result_committed_Hogga_G1_t0,result_committed_Hogga_G1_t1,result_committed_Hogga_G1_t2,result_committed_Hogga_G1_t3,result_committed_Hogga_G1_t4,result_committed_Hogga_G1_t5,result_committed_Hogga_G1_t6
0,2923,0.060042,0.040709,0.019306,0.010764,0.010996,0.009382,0.005569
1,2924,0.908964,0.898183,0.908454,0.906336,0.895133,0.920287,0.958300
2,2925,0.999655,0.999730,0.999717,0.999745,0.999693,0.999765,0.999823


Total minutes: 89.3


## Export professor notebook

In [13]:
# Self-contained, final-only notebook. No old artifacts/src files required when grading.
document=json.loads((ROOT/'notebooks/18_multiscale_boosting.ipynb').read_text())
keep={'resources','config','pipeline','physics','fit_helpers','competition','paths','runner','build'}
short_cells=[]
for c in document['cells']:
    if c.get('id') not in keep:
        continue
    c=json.loads(json.dumps(c)); c['execution_count']=None; c['outputs']=[]
    source=''.join(c['source'])
    if c['id']=='resources':
        source=source.replace("RUN_PROFILE = 'local'", "RUN_PROFILE = 'professor'")
    if c['id']=='runner':
        source=source.split('def select_pair(')[0]+'\n\ndef final_paths('+source.split('def final_paths(',1)[1]
    if c['id']=='build':
        source='\n'.join(line for line in source.splitlines() if not line.startswith("print('Research:") and not line.startswith("print('Each development"))
    c['source']=source.splitlines(keepends=True);short_cells.append(c)
intro='# Short notebook — multiscale boosted blend\n\n**Henrik Remmen (541747), Group 119 — [119] SINTEF Databreach**\n\nRun all cells offline beside the supplied raw data and mapping/sample CSVs. Four CPU threads; no restored model predictions. Hyperparameters selected in notebook 18 on 2021, checked on reused 2022 history. Only 28 final unit fits; no development search. Actual cold runtime must be measured against 12 hours / 32 GB.\n'
short_cells.insert(0,dict(cell_type='markdown',id='intro',metadata={},source=intro.splitlines(keepends=True)))
source='selection = '+repr(selection)+'\npath, submission = final_paths(runner, selection)\nprint("Elapsed minutes:",(time.perf_counter()-NOTEBOOK_STARTED)/60)\n'
short_cells.append(dict(cell_type='code',id='final',metadata={},execution_count=None,outputs=[],source=source.splitlines(keepends=True)))
document['cells']=short_cells
short_path=runner.directory/'Short_notebook_multiscale_v5.ipynb'
short_path.write_text(json.dumps(document,indent=1,ensure_ascii=False)+'\n',encoding='utf-8')
print('Professor notebook:',short_path)
print('Upload candidate:',submission_path)

Professor notebook: /Users/henrik/Documents/H26/Moderne Maksinlæring/sintef_project/artifacts/multiscale_boosting_v5/Short_notebook_multiscale_v5.ipynb
Upload candidate: /Users/henrik/Documents/H26/Moderne Maksinlæring/sintef_project/submissions/multiscale_boosting_v5.csv


## Files and interpretation

- Upload candidate: `submissions/multiscale_boosting_v5.csv`.
- Individual models: `submissions/xgb_multiscale_v5.csv`, `submissions/lgb_multiscale_v5.csv`.
- Selection, feature manifest, development scores, 2022 diagnostics, per-unit gains and
  runtimes: `artifacts/multiscale_boosting_v5/`.
- Professor version: `artifacts/multiscale_boosting_v5/Short_notebook_multiscale_v5.ipynb`.

Inspect `development_scores.csv`: compare 1x v4 against 1x path to isolate features,
then compare 1x/1.5x/2x within each variant to assess additional boosting. Gain rankings
are descriptive, not causal proof of a feature's value. Check audit errors by unit,
month and horizon day. Do not claim a leaderboard improvement before submitting.

Research details: `RESEARCH_MULTISCALE_V5.md` in the development workspace.

Primary references: [XGBoost tuning](https://xgboost.readthedocs.io/en/stable/tutorials/param_tuning.html),
[LightGBM tuning](https://lightgbm.readthedocs.io/en/stable/Parameters-Tuning.html),
[SINTEF generator physics](https://docs.shop.sintef.energy/objects/generator/generator.html),
[scikit-learn calibration](https://scikit-learn.org/stable/modules/calibration.html).
These motivate hypotheses and safeguards; they do not establish performance on this task.